In [2]:
import tkinter as tk
from tkinter import messagebox
import random
import copy

max_stages = 7


# ──────────────────────────────────────────────
# Game Logic
# ──────────────────────────────────────────────

class ColorSortingGame:
    def __init__(self, stage):
        self.stage = stage
        self.tubes = self.generate_tubes(stage)
        self.history = []

    def generate_tubes(self, stage):
        all_colors = ["red", "green", "blue", "yellow", "orange",
                      "purple", "cyan", "magenta", "brown"]

        num_colors = min(stage + 2, len(all_colors))   
        tube_size  = 4
        num_empty  = 1 + (stage // 2)
        total_tubes = num_colors + num_empty

        color_pool = [all_colors[i] for i in range(num_colors)] * tube_size
        random.shuffle(color_pool)

        tubes = [[] for _ in range(total_tubes)]
        for t in range(num_colors):
            tubes[t] = color_pool[t * tube_size:(t + 1) * tube_size]
        return tubes

    def move(self, src, dest):
        if src == dest or not self.tubes[src]:
            return False
        if len(self.tubes[dest]) >= 4:
            return False
        if self.tubes[dest] and self.tubes[dest][-1] != self.tubes[src][-1]:
            return False

        self.history.append(copy.deepcopy(self.tubes))
        color = self.tubes[src].pop()
        self.tubes[dest].append(color)
        return True

    def undo(self):
        if self.history:
            self.tubes = self.history.pop()
            return True
        return False

    def is_solved(self):
        return all(
            len(tube) == 0 or (len(tube) == 4 and len(set(tube)) == 1)
            for tube in self.tubes
        )


# ──────────────────────────────────────────────
# DFS Hint Solver
# ──────────────────────────────────────────────

def get_possible_moves(tubes):
    moves = []
    n = len(tubes)
    for i in range(n):
        if not tubes[i]:
            continue
        for j in range(n):
            if i != j and len(tubes[j]) < 4:
                if not tubes[j] or tubes[j][-1] == tubes[i][-1]:
                    moves.append((i, j))
    return moves

def apply_move(tubes, src, dest):
    new_tubes = copy.deepcopy(tubes)
    new_tubes[dest].append(new_tubes[src].pop())
    return new_tubes

def is_goal_state(tubes):
    return all(
        len(tube) == 0 or (len(tube) == 4 and len(set(tube)) == 1)
        for tube in tubes
    )

def dfs_solver(tubes, visited=None, path=None, depth=0, max_depth=50):
    if visited is None:
        visited = set()
    if path is None:
        path = []

    state_key = tuple(tuple(t) for t in tubes)
    if state_key in visited or depth > max_depth:
        return None
    visited.add(state_key)

    if is_goal_state(tubes):
        return path

    for move in get_possible_moves(tubes):
        new_tubes = apply_move(tubes, *move)
        result = dfs_solver(new_tubes, visited, path + [move], depth + 1, max_depth)
        if result is not None:
            return result

    return None


# ──────────────────────────────────────────────
# Application Controller
# ──────────────────────────────────────────────

class App:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Color Sorting Game")
        self.player_name  = ""
        self.player_score = 0
        self._show_main_screen()
        self.root.mainloop()

    # ── helpers ──────────────────────────────

    def _clear(self):
        """Destroy all widgets in the root window."""
        for w in self.root.winfo_children():
            w.destroy()

    # ── screens ──────────────────────────────

    def _show_main_screen(self):
        self._clear()
        self.root.geometry("300x200")
        self.root.title("Color Sorting Game")

        tk.Label(self.root, text="Color Sorting Game",
                 font=("Arial", 16, "bold")).pack(pady=15)
        tk.Label(self.root, text="Enter your name:").pack()
        name_entry = tk.Entry(self.root)
        name_entry.pack(pady=5)

        def start():
            name = name_entry.get().strip()
            if name:
                self.player_name  = name
                self.player_score = 0
                self._launch_stage(1)
            else:
                messagebox.showwarning("Input Error", "Please enter your name.")

        tk.Button(self.root, text="Play", command=start).pack(pady=10)

    def _launch_stage(self, stage_num):
        self._clear()

        game = ColorSortingGame(stage_num)
        num_tubes = len(game.tubes)


        win_width  = max(700, num_tubes * 80 + 40)
        win_height = 320
        self.root.geometry(f"{win_width}x{win_height}")
        self.root.title(f"Stage {stage_num} – Color Sorting Game")


        score_var = tk.StringVar(value=f"Player: {self.player_name}   Score: {self.player_score}")
        tk.Label(self.root, textvariable=score_var, font=("Arial", 11)).pack(pady=(8, 0))

        tubes_frame = tk.Frame(self.root)
        tubes_frame.pack(pady=10)

        selected      = []        
        canvas_refs   = []    

        def draw_tubes(highlighted=None):
            for w in tubes_frame.winfo_children():
                w.destroy()
            canvas_refs.clear()

            for idx, tube in enumerate(game.tubes):
                border_color = "gold" if idx == highlighted else "grey"
                canvas = tk.Canvas(tubes_frame, width=60, height=180,
                                   bg="lightgrey",
                                   highlightthickness=3,
                                   highlightbackground=border_color)
                canvas.grid(row=0, column=idx, padx=8)
                canvas_refs.append(canvas)

                y = 180
                for color in tube:
                    canvas.create_rectangle(5, y - 40, 55, y,
                                            fill=color, outline="black")
                    y -= 40

                canvas.bind("<Button-1>",
                            lambda e, i=idx: on_tube_click(i))

        def on_tube_click(idx):
            if not selected:
                if game.tubes[idx]:         
                    selected.append(idx)
                    draw_tubes(highlighted=idx)
            else:
                src = selected.pop()
                if src == idx:
                    draw_tubes()
                    return
                if game.move(src, idx):
                    draw_tubes()
                    score_var.set(f"Player: {self.player_name}   Score: {self.player_score}")
                    if game.is_solved():
                        self.player_score += 1
                        score_var.set(f"Player: {self.player_name}   Score: {self.player_score}")
                        self._ask_next_stage(stage_num)
                else:
                    selected.append(src)
                    messagebox.showinfo("Invalid Move", "That move isn't allowed.")
                    draw_tubes(highlighted=src)

        def undo_action():
            if game.undo():
                draw_tubes()
            else:
                messagebox.showinfo("Undo", "Nothing to undo!")

        def restart_stage():
            self._launch_stage(stage_num)

        def show_hint():
            hint = dfs_solver(game.tubes)
            if hint:
                src, dest = hint[0]
                messagebox.showinfo("Hint",
                    f"Try moving from tube {src + 1} to tube {dest + 1}")
            else:
                messagebox.showinfo("Hint", "No solution found from here — try undoing a move.")

        ctrl = tk.Frame(self.root)
        ctrl.pack(pady=5)
        tk.Button(ctrl, text="Undo",          command=undo_action).grid(row=0, column=0, padx=5)
        tk.Button(ctrl, text="Restart Stage", command=restart_stage).grid(row=0, column=1, padx=5)
        tk.Button(ctrl, text="Show Hint",     command=show_hint,
                  bg="lightyellow").grid(row=0, column=2, padx=5)

        draw_tubes()

    def _ask_next_stage(self, stage_num):
        go_next = messagebox.askyesno("Stage Complete! 🎉",
            f"Great job, {self.player_name}!\n"
            f"Score: {self.player_score}\n\n"
            "Go to next stage?")
        if go_next:
            if stage_num < max_stages:
                self._launch_stage(stage_num + 1)
            else:
                self._show_champion_screen()
        else:
            self._show_main_screen()

    
    def _show_champion_screen(self):
        self._clear()
        self.root.geometry("400x220")
        self.root.title("Champion!")

        tk.Label(self.root, text="🏆 Champion! 🏆",
                 font=("Arial", 20, "bold")).pack(pady=15)
        tk.Label(self.root,
                 text=f"Well done, {self.player_name}!\nFinal Score: {self.player_score}",
                 font=("Arial", 13)).pack(pady=5)
        tk.Label(self.root,
                 text="More stages coming soon!",
                 font=("Arial", 10), fg="grey").pack()
        tk.Button(self.root, text="Play Again",
                  command=self._show_main_screen).pack(pady=15)


# ──────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────

if __name__ == "__main__":
    App()